In [5]:
# Test EmbeddingCNN 1D - Small Model Verification
# 
# Quick test to verify the EmbeddingCNN 1D model works correctly
# Uses small model configuration for fast verification

import sys
import os
sys.path.append('../../../')

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Import our custom models
from embedding_cnn_models import (
    EmbeddingCNN, BinaryClassificationDataset, count_parameters
)

# Force reload the module to get latest changes
import importlib
import embedding_cnn_models
importlib.reload(embedding_cnn_models)
from embedding_cnn_models import (
    EmbeddingCNN, BinaryClassificationDataset, count_parameters
)

print("✅ Imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


✅ Imports successful!
PyTorch version: 2.8.0
CUDA available: False
Using device: cpu


In [6]:
# Load small subset of data for quick testing
print("📊 Loading data...")

data_binary = pd.read_csv('../../../data/processed/ProSeq_binary_classification.csv')
data_binary = data_binary[['binary_classification', 'ProSeq']]

# Filter to sequences >= 600
sequence_lengths = data_binary['ProSeq'].str.len()
data_filtered = data_binary[sequence_lengths >= 600].copy()

# Take small subset for quick testing (500 samples)
data_small = data_filtered.sample(n=500, random_state=42)

print(f"Original dataset: {len(data_binary):,} samples")
print(f"Filtered (>=600): {len(data_filtered):,} samples")
print(f"Test subset: {len(data_small):,} samples")

# Check class distribution
class_counts = data_small['binary_classification'].value_counts()
print(f"\nClass distribution in test subset:")
for class_label, count in class_counts.items():
    print(f"  Class {class_label}: {count} samples ({count/len(data_small)*100:.1f}%)")

# Split data
train_data, test_data = train_test_split(
    data_small, test_size=0.3, random_state=42, 
    stratify=data_small['binary_classification']
)
train_data, val_data = train_test_split(
    train_data, test_size=0.3, random_state=42, 
    stratify=train_data['binary_classification']
)

print(f"\nData splits:")
print(f"  Train: {len(train_data)} samples")
print(f"  Validation: {len(val_data)} samples") 
print(f"  Test: {len(test_data)} samples")

# Create datasets and dataloaders
train_dataset = BinaryClassificationDataset(train_data)
val_dataset = BinaryClassificationDataset(val_data)
test_dataset = BinaryClassificationDataset(test_data)

batch_size = 16  # Small batch size for testing
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nDataLoaders created with batch size: {batch_size}")

# Test data loading
sample_batch = next(iter(train_loader))
print(f"Sample batch shapes: Input {sample_batch[0].shape}, Target {sample_batch[1].shape}")
print(f"Input range: [{sample_batch[0].min():.1f}, {sample_batch[0].max():.1f}]")
print(f"Target values: {torch.unique(sample_batch[1])}")


📊 Loading data...
Original dataset: 8,307 samples
Filtered (>=600): 8,302 samples
Test subset: 500 samples

Class distribution in test subset:
  Class 0: 311 samples (62.2%)
  Class 1: 189 samples (37.8%)

Data splits:
  Train: 245 samples
  Validation: 105 samples
  Test: 150 samples

DataLoaders created with batch size: 16
Sample batch shapes: Input torch.Size([16, 600, 5]), Target torch.Size([16])
Input range: [0.0, 1.0]
Target values: tensor([0, 1])


In [7]:
# Create small EmbeddingCNN 1D model for testing
print("Creating Small EmbeddingCNN 1D Model")
print("=" * 45)

# Small model configuration - easy to verify
model = EmbeddingCNN(
    input_channels=5,           # A, T, G, C, N
    sequence_length=600,
    embedding_dims=[128],        # Single embedding: 5 → 16
    conv_filters=[4, 8],      # Small conv layers
    kernel_sizes=[13, 11],        # Standard kernel sizes
    use_2d_conv=False,          # 1D convolution
    dropout=0.2                 # Light dropout
)

# Move to device
model = model.to(device)

# Count parameters
param_count = count_parameters(model)
print(f"Model parameters: {param_count:,}")
print(f"Samples per parameter: {len(train_data) / param_count:.1f}")

# Print model architecture
print(f"\n")
model.log_architecture()

# Test forward pass
print(f"\nTesting forward pass...")
sample_x, sample_y = sample_batch
sample_x = sample_x.to(device)

model.eval()
with torch.no_grad():
    try:
        output = model(sample_x)
        print(f"Forward pass successful!")
        print(f"  Input shape: {sample_x.shape}")
        print(f"  Output shape: {output.shape}")
        print(f"  Output range: [{output.min():.3f}, {output.max():.3f}]")
        print(f"  Output sample: {output[:5].cpu().numpy()}")
    except Exception as e:
        print(f"Forward pass failed: {e}")

# Test gradient computation
print(f"\nTesting gradient computation...")
model.train()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

try:
    # Forward pass
    output = model(sample_x)
    target = sample_y.float().to(device)
    
    # Compute loss
    loss = criterion(output, target)
    print(f"Loss computation successful!")
    print(f"  Loss value: {loss.item():.4f}")
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    
    # Check gradients
    total_grad_norm = 0
    param_count = 0
    for param in model.parameters():
        if param.grad is not None:
            total_grad_norm += param.grad.data.norm(2).item() ** 2
            param_count += 1
    
    total_grad_norm = total_grad_norm ** 0.5
    print(f"Gradient computation successful!")
    print(f"  Total gradient norm: {total_grad_norm:.4f}")
    print(f"  Parameters with gradients: {param_count}")
    
    # Optimizer step
    optimizer.step()
    print(f"Optimizer step successful!")
    
except Exception as e:
    print(f"Gradient computation failed: {e}")

print(f"\nModel verification complete!")


Creating Small EmbeddingCNN 1D Model
Model parameters: 169,861
Samples per parameter: 0.0


Model Architecture:
  Input: One-hot encoded DNA (batch, 600, 5)
  Embedding 1: Linear(5 -> 128) + ReLU + Dropout
  Conv1D 1: (128 -> 4, kernel=13) + BatchNorm + ReLU + MaxPool
  Conv1D 2: (4 -> 8, kernel=11) + BatchNorm + ReLU + MaxPool
  Classifier: Linear layers -> Sigmoid
  Output: Binary probability
  Total Parameters: 169,861

Testing forward pass...
Forward pass successful!
  Input shape: torch.Size([16, 600, 5])
  Output shape: torch.Size([16])
  Output range: [0.519, 0.520]
  Output sample: [0.51928014 0.5192138  0.5200277  0.5195779  0.5191936 ]

Testing gradient computation...
Loss computation successful!
  Loss value: 0.7180
Gradient computation successful!
  Total gradient norm: 1.6407
  Parameters with gradients: 16
Optimizer step successful!

Model verification complete!


In [ ]:
# Quick training test (30 epochs)
print("Quick Training Test (30 epochs)")
print("=" * 40)

def quick_train(model, num_epochs=30):
    """Quick training function for verification."""
    
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    train_losses = []
    val_losses = []
    val_accuracies = []
    
    print(f"Training for {num_epochs} epochs...")
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        epoch_train_loss = 0.0
        
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.float().to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_train_loss += loss.item()
        
        # Validation phase
        model.eval()
        epoch_val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.float().to(device)
                
                outputs = model(batch_x)
                loss = criterion(outputs, batch_y)
                epoch_val_loss += loss.item()
                
                # Calculate accuracy
                predicted = (outputs > 0.5).float()
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        
        # Calculate averages
        avg_train_loss = epoch_train_loss / len(train_loader)
        avg_val_loss = epoch_val_loss / len(val_loader)
        val_accuracy = correct / total
        
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)
        
        print(f"Epoch {epoch+1}/{num_epochs}: "
              f"Train Loss: {avg_train_loss:.4f}, "
              f"Val Loss: {avg_val_loss:.4f}, "
              f"Val Acc: {val_accuracy:.4f}")
    
    return train_losses, val_losses, val_accuracies

# Run quick training
try:
    train_losses, val_losses, val_accuracies = quick_train(model, num_epochs=30)
    print(f"\nTraining completed successfully!")
    print(f"Final validation accuracy: {val_accuracies[-1]:.4f}")
    
    # Plot training curves
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    epochs = range(1, len(train_losses) + 1)
    plt.plot(epochs, train_losses, 'b-', label='Train Loss')
    plt.plot(epochs, val_losses, 'r-', label='Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(epochs, val_accuracies, 'g-o', label='Val Accuracy')
    plt.title('Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Training failed: {e}")

print(f"\nQuick training test complete!")


Quick Training Test (30 epochs)
Training for 30 epochs...
Epoch 1/30: Train Loss: 0.6803, Val Loss: 0.6849, Val Acc: 0.6190
Epoch 2/30: Train Loss: 0.6724, Val Loss: 0.6539, Val Acc: 0.6190
Epoch 3/30: Train Loss: 0.6430, Val Loss: 0.6610, Val Acc: 0.6190
Epoch 4/30: Train Loss: 0.6389, Val Loss: 0.6577, Val Acc: 0.6286
Epoch 5/30: Train Loss: 0.6191, Val Loss: 0.6567, Val Acc: 0.6095
Epoch 6/30: Train Loss: 0.6101, Val Loss: 0.6660, Val Acc: 0.6190
Epoch 7/30: Train Loss: 0.5671, Val Loss: 0.6617, Val Acc: 0.6190
Epoch 8/30: Train Loss: 0.5241, Val Loss: 0.8179, Val Acc: 0.4571
Epoch 9/30: Train Loss: 0.5618, Val Loss: 0.6582, Val Acc: 0.5905
Epoch 10/30: Train Loss: 0.4445, Val Loss: 0.7289, Val Acc: 0.6286
Epoch 11/30: Train Loss: 0.3967, Val Loss: 0.7392, Val Acc: 0.6381
Epoch 12/30: Train Loss: 0.4449, Val Loss: 0.7079, Val Acc: 0.6000
